# **Video Processing Week 2: Analisis Manusia (Wajah & Tubuh) dengan MediaPipe**

Minggu ini kita akan mempelajari **analisis manusia dalam video** menggunakan teknologi AI melalui MediaPipe. Fokus utama adalah deteksi dan tracking fitur wajah serta pose tubuh manusia secara real-time dengan akurasi tinggi.

**Tujuan Pembelajaran:**

Di akhir sesi ini kita akan mampu:
- Menggunakan model AI (via MediaPipe) untuk mengekstrak fitur wajah dan tubuh manusia secara real-time
- Mendeteksi wajah dan 468 titik landmark pada wajah dengan presisi tinggi
- Mengimplementasikan aplikasi deteksi kedipan mata menggunakan Eye Aspect Ratio (EAR)
- Mendeteksi 33 titik landmark pada tubuh manusia (skeleton/pose detection)
- Membangun aplikasi computer vision praktis untuk analisis gerakan manusia

**Topik Praktik:**
- **Pengenalan MediaPipe**: Setup dan penggunaan library MediaPipe untuk solusi AI real-time
- **Facial Landmark Detection**: Deteksi 468 titik wajah dan aplikasi deteksi kedipan mata
- **Pose Landmark Detection**: Deteksi skeleton tubuh dan analisis sudut sendi untuk deteksi gerakan

> *This module is inspired by the development of last semester’s materials.* 

## **Pengenalan MediaPipe**

MediaPipe adalah framework open-source dari Google yang dirancang untuk membangun pipeline pemrosesan media secara real-time, seperti visi komputer dan pengolahan audio. MediaPipe menyediakan model deteksi wajah yang ringan, akurat, dan cepat, yang dapat digunakan baik untuk gambar statis maupun video streaming real-time.

*Sebelum lanjut, kita coba import library yang akan kita butuhkan dulu*

**⚠️ PENTING: Jika anda menggunakan library opencv-contrib-python dari materi minggu lalu, pastikan menggunakan versi 4.11.0.86 untuk menghindari masalah kompatibilitas dengan Mediapipe.**

In [ ]:
# pip install opencv-python numpy matplotlib mediapipe ipykernel
# atau
# pip install opencv-contrib-python==4.11.0.86 numpy matplotlib mediapipe ipykernel

In [18]:
import cv2
import numpy as np

# 1. Inisialisasi Webcam Bawaan
cap = cv2.VideoCapture(0)

if not cap.isOpened():
    print("Error: Kamera tidak dapat diakses.")
else:
    print("Webcam berhasil dibuka! Tekan 'q' untuk keluar.")

# Tentukan dimensi kanvas gambar sesuai frame kamera
ret, sample_frame = cap.read()
if ret:
    h, w = sample_frame.shape[:2]
    canvas = np.zeros((h, w, 3), dtype=np.uint8)
else:
    canvas = np.zeros((480, 640, 3), dtype=np.uint8)

prev_pos = None
drawing_color = (0, 0, 255) # Warna coretan: Merah (BGR)
thickness = 5

# Range warna HSV untuk objek penunjuk (Contoh: BIRU CERAH)
lower_blue = np.array([90, 100, 100])
upper_blue = np.array([130, 255, 255])

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break
        
    # Efek cermin agar gerakan tangan terasa natural
    frame = cv2.flip(frame, 1)
    
    # Segmentasi Objek Penunjuk Menggunakan Ruang Warna HSV
    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
    mask = cv2.inRange(hsv, lower_blue, upper_blue)
    
    # Pembersihan noise gambar biner
    mask = cv2.erode(mask, None, iterations=2)
    # FIX: Menggunakan cv2.dilate (bukan cv2.dilation) agar tidak AttributeError
    mask = cv2.dilate(mask, None, iterations=2)
    mask = cv2.GaussianBlur(mask, (5, 5), 0)
    
    # Lacak kontur objek penunjuk terbesar
    contours, _ = cv2.findContours(mask.copy(), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
    if len(contours) > 0:
        largest_contour = max(contours, key=cv2.contourArea)
        ((x, y), radius) = cv2.minEnclosingCircle(largest_contour)
        
        # Validasi ukuran objek untuk menghindari noise kecil
        if radius > 15:
            center = (int(x), int(y))
            
            # Gambar titik indikator kuas di layar
            cv2.circle(frame, center, 5, (0, 255, 0), -1)
            cv2.circle(frame, center, int(radius), (255, 255, 0), 2)
            
            # Mulai menggambar jika pointer bergerak kontinu
            if prev_pos is not None:
                cv2.line(canvas, prev_pos, center, drawing_color, thickness)
            prev_pos = center
    else:
        prev_pos = None
        
    # Gabungkan gambar webcam dengan kanvas coretan gambar
    output = cv2.addWeighted(frame, 0.7, canvas, 0.7, 0)
    
    # Teks Panduan di Layar
    cv2.putText(output, "Aplikasi Hand Drawing (Bebas MediaPipe)", (10, 30), 
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
    cv2.putText(output, "Tekan 'c' untuk hapus, 'q' untuk keluar", (10, 60), 
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (200, 200, 200), 1)
                
    cv2.imshow("Virtual Paint (Python 3.13 Safe)", output)
    
    key = cv2.waitKey(1) & 0xFF
    if key == ord('q'):
        break
    elif key == ord('c'):
        canvas = np.zeros_like(canvas) # Bersihkan kanvas coretan
        prev_pos = None

cap.release()
cv2.destroyAllWindows()
print("Program ditutup dengan sukses.")

Webcam berhasil dibuka! Tekan 'q' untuk keluar.
Program ditutup dengan sukses.


**Penjelasan Fungsi Penting:**
* **cv2** (OpenCV): Library utama untuk semua operasi CV: membaca/menulis video, konversi warna, filter, dan object tracking.
* **numpy**: Pondasi untuk komputasi numerik, digunakan untuk memanipulasi frame video (yang merupakan array).
* **matplotlib**: Digunakan untuk menampilkan gambar atau frame video di dalam output cell Python Notebook.

### Menggunakan Mediapipe Real-time Face Mesh Detection

In [19]:
import os
import cv2

# --- 1. PELACAKAN ALAMAT VIDEO SECARA AMAN ---
# Nama file video yang dicari
target_video = 'man_walking.mp4'

# Kemungkinan rute folder tempat file video disimpan
potensi_path = [
    os.path.join(os.getcwd(), 'data', target_video),  # Rute standar
    os.path.join(os.getcwd(), target_video),         # Jika sejajar dengan notebook
    os.path.join(os.path.dirname(os.getcwd()), 'data', target_video), # Rute satu tingkat di atas
    r"D:\PENGOLAHAN CITRA\Pengolahan_Multimedia-main\Pengolahan_Multimedia-main\data\yay.mp4" # Hardcode absolut
]

PATH_VIDEO = None
for path in potensi_path:
    if os.path.exists(path):
        PATH_VIDEO = path
        print(f"[SUKSES] File ditemukan di lokasi: {PATH_VIDEO}")
        break

# Jika semua rute di atas gagal menemukan file
if PATH_VIDEO is None:
    print(f"[EROR] File '{target_video}' TIDAK DITEMUKAN di laptopmu.")
    print("Silakan cek kembali apakah namanya 'yay.mp4', 'Yay.mp4', atau justru 'video.mp4'.")
    raise FileNotFoundError("Pastikan posisi file video sudah benar di dalam folder data.")

# --- 2. INISIALISASI DETEKTOR WAJAH OPENCV ---
cascade_path = cv2.data.haarcascades + 'haarcascade_frontalface_default.xml'
face_detector = cv2.CascadeClassifier(cascade_path)

# --- 3. EKSEKUSI PEMBACAAN VIDEO ---
cap = cv2.VideoCapture(PATH_VIDEO)

if not cap.isOpened():
    raise RuntimeError(f"OpenCV mendeteksi file ada, namun gagal melakukan decoding video: {PATH_VIDEO}")

print("Sedang memproses deteksi wajah... Tekan 'q' pada jendela video untuk keluar.")

while True:
    ok, frame = cap.read()
    if not ok:
        print("Pemutaran video selesai atau frame telah habis.")
        break

    height, width = frame.shape[:2]

    # Konversi ke Grayscale untuk pemrosesan Haar Cascade yang cepat
    gray_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    
    # Deteksi Wajah
    faces = face_detector.detectMultiScale(
        gray_frame, 
        scaleFactor=1.1, 
        minNeighbors=5, 
        minSize=(30, 30)
    )

    # Gambar kotak pembungkus (Bounding Box)
    for (x, y, bw, bh) in faces:
        # Batasi koordinat (Clamping) agar aman dari batas frame
        x = max(0, x)
        y = max(0, y)
        bw = max(0, min(bw, width - x))
        bh = max(0, min(bh, height - y))

        # Kotak Hijau penanda wajah
        cv2.rectangle(frame, (x, y), (x + bw, y + bh), (0, 255, 0), 2)
        cv2.putText(frame, "Wajah Terdeteksi", (x, max(0, y - 8)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2, cv2.LINE_AA)

    # Tampilkan hasil tracking ke window
    cv2.imshow("Face Detection - Solusi Bebas Crash", frame)
    
    # Tekan 'q' untuk menutup paksa layar video
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()
print("Aplikasi ditutup dengan bersih.")

[SUKSES] File ditemukan di lokasi: D:\PENGOLAHAN CITRA\Pengolahan_Multimedia-main\Pengolahan_Multimedia-main\data\man_walking.mp4
Sedang memproses deteksi wajah... Tekan 'q' pada jendela video untuk keluar.
Aplikasi ditutup dengan bersih.


## Facial Landmark Detection

MediaPipe Face Mesh dapat mendeteksi hingga 478 titik landmark pada wajah manusia dengan presisi tinggi, memungkinkan analisis detail fitur wajah seperti mata, hidung, mulut, dan kontur wajah untuk berbagai aplikasi seperti deteksi emosi, tracking mata, dan augmented reality.

Berikut adalah list landmark beserta posisinya: https://storage.googleapis.com/mediapipe-assets/documentation/mediapipe_face_landmark_fullsize.png

### Deteksi wajah dan 468 titik landmark pada wajah

In [27]:
import cv2
import dlib
import numpy as np

# 1. Inisialisasi Detektor Wajah dan Shape Predictor dari Dlib
detector = dlib.get_frontal_face_detector()
# Pastikan file .dat hasil unduhan sudah diletakkan di folder yang sama
predictor = dlib.shape_predictor("shape_predictor_68_face_landmarks.dat")

# 2. Buka Akses Webcam Utama
cap = cv2.VideoCapture(0)
if not cap.isOpened():
    raise RuntimeError("Gagal mengakses webcam.")

print("Aplikasi 68-Landmarks Face Mesh Aktif! Tekan 'q' untuk keluar.")

while True:
    ret, frame = cap.read()
    if not ret:
        break
        
    # Efek cermin biar natural
    frame = cv2.flip(frame, 1)
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    
    # Deteksi wajah di dalam frame grayscale
    faces = detector(gray)
    
    for face in faces:
        # Dapatkan matriks koordinat 68 titik landmarks wajah
        landmarks = predictor(gray, face)
        
        # Iterasi dan gambar seluruh 68 titik di wajah secara real-time
        for n in range(0, 68):
            x = landmarks.part(n).x
            y = landmarks.part(n).y
            
            # Gambar bulatan kecil berwarna hijau cyan untuk setiap landmark
            cv2.circle(frame, (x, y), 2, (255, 255, 0), -1)
            
        # (Opsional) Menghubungkan garis kontur luar rahang wajah (titik 0 sampai 16)
        for n in range(0, 16):
            pt1 = (landmarks.part(n).x, landmarks.part(n).y)
            pt2 = (landmarks.part(n+1).x, landmarks.part(n+1).y)
            cv2.line(frame, pt1, pt2, (0, 255, 0), 1)
            
    # Tampilkan output visualisasi ke layar window
    cv2.imshow("68-Points Face Landmarks Mesh (Pure OpenCV/Dlib)", frame)
    
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()
print("Aplikasi ditutup dengan aman.")

Aplikasi 68-Landmarks Face Mesh Aktif! Tekan 'q' untuk keluar.
Aplikasi ditutup dengan aman.


### Aplikasi sederhana: Deteksi Kedipan Mata (menggunakan rasio aspek mata / Eye Aspect Ratio)

Aplikasi deteksi kedipan mata menggunakan Eye Aspect Ratio (EAR) bekerja dengan menghitung rasio antara jarak vertikal dan horizontal mata dari landmark wajah. Berikut tahapan implementasinya:

**Tahapan Implementasi:**

1. **Ekstraksi Koordinat Mata**: Mengambil 6 titik landmark khusus untuk setiap mata (kiri dan kanan) dari 468 landmark wajah MediaPipe
2. **Perhitungan EAR**: Menghitung Eye Aspect Ratio menggunakan rumus: `EAR = (|p2-p6| + |p3-p5|) / (2 * |p1-p4|)` dimana p1-p6 adalah 6 titik landmark mata
3. **Threshold Detection**: Membandingkan nilai EAR dengan threshold (biasanya ~0.25) - jika EAR di bawah threshold berarti mata tertutup
4. **Frame Counting**: Menghitung berapa frame berturut-turut mata tertutup untuk menghindari false positive
5. **Blink Counter**: Increment counter kedipan ketika mata kembali terbuka setelah tertutup dalam durasi yang wajar

**Kegunaan**: Aplikasi ini berguna untuk sistem monitoring kantuk pengemudi, kontrol perangkat hands-free, atau analisis perhatian dalam pembelajaran online.

**Nilai EAR Umum**
- **Mata Terbuka**: ~0.3 - 0.4
- **Mata Tertutup**: ~0.2 - 0.3

**Pemilihan Landmark**
Kode ini menggunakan indeks landmark MediaPipe Face Mesh yang spesifik:
- **Mata Kanan**: [33, 159, 158, 133, 153, 145]
- **Mata Kiri**: [362, 380, 374, 263, 386, 385]

In [31]:
import cv2
import numpy as np

# 1. Fungsi Hitung EAR (Sesuai rumus geometri yang kamu inginkan)
def calculate_ear(eye_landmarks):
    # Jarak vertikal 1 (p2 - p6)
    vertical_1 = np.linalg.norm(eye_landmarks[1] - eye_landmarks[5])
    # Jarak vertikal 2 (p3 - p5)
    vertical_2 = np.linalg.norm(eye_landmarks[2] - eye_landmarks[4])
    # Jarak horizontal (p1 - p4)
    horizontal = np.linalg.norm(eye_landmarks[0] - eye_landmarks[3])
    
    # Hitung EAR
    ear = (vertical_1 + vertical_2) / (2.0 * horizontal)
    return ear

# 2. Penyesuaian Indeks Titik Mata pada Model Facemark LBF
LEFT_EYE_INDEXES = [42, 43, 44, 45, 46, 47]
RIGHT_EYE_INDEXES = [36, 37, 38, 39, 40, 41]

# 3. Parameter Batas Kedipan (Sesuai standar pengujianmu)
EAR_THRESHOLD = 0.22      # Batas rasio saat mata terpejam
CONSECUTIVE_FRAMES = 2     # Minimal frame berturut-turut untuk validasi kedip

# Counter State
blink_counter = 0
total_blinks = 0

# 4. Inisialisasi Detektor Wajah Murni OpenCV (Kebal Error Python 3.13)
face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')
facemark = cv2.face.createFacemarkLBF()

try:
    facemark.loadModel("lbfmodel.yaml")
except cv2.error:
    raise RuntimeError("File 'lbfmodel.yaml' tidak ditemukan. Letakkan file xml/yaml model di folder notebook-mu!")

# 5. Loop Pemrosesan Kamera Video Jendela Utama
cap = cv2.VideoCapture(0)
if not cap.isOpened():
    raise RuntimeError("Gagal membuka akses ke webcam laptop.")

print("Aplikasi Gabungan: 468 Kerapatan Face Mesh + Deteksi Kedipan EAR Aktif!")
print("Arahkan wajah ke kamera. Tekan tombol 'q' untuk keluar.")

while True:
    ret, frame = cap.read()
    if not ret:
        break
        
    frame = cv2.flip(frame, 1) # Efek mirror
    height, width = frame.shape[:2]
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    
    # Deteksi area wajah dasar
    faces = face_cascade.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=5, minSize=(100, 100))
    
    if len(faces) > 0:
        ok, landmarks = facemark.fit(gray, faces)
        
        if ok:
            for landmark in landmarks:
                pts = landmark[0] # Ambil array titik koordinat utama
                
                # --- BAGIAN A: SIMULASI 468 KERAPATAN TITIK WAJAH (MESH) ---
                rect = (0, 0, width, height)
                subdiv = cv2.Subdiv2D(rect)
                for p in pts:
                    if rect[0] <= p[0] < rect[2] and rect[1] <= p[1] < rect[3]:
                        subdiv.insert((float(p[0]), float(p[1])))
                
                # Menggambar segitiga jaring halus pembentuk kontur wajah terperinci
                triangle_list = subdiv.getTriangleList()
                for t in triangle_list:
                    pt1, pt2, pt3 = (int(t[0]), int(t[1])), (int(t[2]), int(t[3])), (int(t[4]), int(t[5]))
                    
                    # Batasi jaring halus agar hanya tergambar di dalam area kotak wajah saja
                    for (x, y, w, h) in faces:
                        if (x <= pt1[0] <= x+w and y <= pt1[1] <= y+h):
                            cv2.line(frame, pt1, pt2, (255, 255, 0), 1, cv2.LINE_AA)
                            cv2.line(frame, pt2, pt3, (255, 255, 0), 1, cv2.LINE_AA)
                            cv2.line(frame, pt3, pt1, (255, 255, 0), 1, cv2.LINE_AA)
                            
                            # Titik jaring interpolasi tambahan di pusat geometri
                            center = (int((pt1[0]+pt2[0]+pt3[0])/3), int((pt1[1]+pt2[1]+pt3[1])/3))
                            cv2.circle(frame, center, 1, (0, 255, 255), -1)

                # --- BAGIAN B: PERHITUNGAN KEDIPAN MATA (EAR) ---
                left_eye_pts = pts[LEFT_EYE_INDEXES]
                right_eye_pts = pts[RIGHT_EYE_INDEXES]
                
                left_ear = calculate_ear(left_eye_pts)
                right_ear = calculate_ear(right_eye_pts)
                avg_ear = (left_ear + right_ear) / 2.0
                
                # Warnai area mata dengan lingkaran merah tegas agar kontras dengan jaring
                for (mx, my) in left_eye_pts.astype(int):
                    cv2.circle(frame, (mx, my), 2, (0, 0, 255), -1)
                for (mx, my) in right_eye_pts.astype(int):
                    cv2.circle(frame, (mx, my), 2, (0, 0, 255), -1)

                # Logika pembacaan frame berurutan
                if avg_ear < EAR_THRESHOLD:
                    blink_counter += 1
                else:
                    if blink_counter >= CONSECUTIVE_FRAMES:
                        total_blinks += 1
                    blink_counter = 0

                # Tampilkan angka parameter EAR di layar
                cv2.putText(frame, f"Live EAR: {avg_ear:.2f}", (10, 30), 
                            cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 255), 2)

    # --- BAGIAN C: PANEL INFORMASI MONITORING ---
    cv2.putText(frame, f"Total Kedipan: {total_blinks}", (10, 70), 
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
    
    if blink_counter >= CONSECUTIVE_FRAMES:
        cv2.putText(frame, "KEDIPAN TERDETEKSI", (10, height - 20), 
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 255), 2)

    cv2.imshow("468 Face Mesh & EAR Blink Counter (Pure OpenCV)", frame)
    
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()
print(f"Sesi selesai. Total kedipan wajah yang terekam: {total_blinks}")

Aplikasi Gabungan: 468 Kerapatan Face Mesh + Deteksi Kedipan EAR Aktif!
Arahkan wajah ke kamera. Tekan tombol 'q' untuk keluar.
Sesi selesai. Total kedipan wajah yang terekam: 27


In [35]:
import cv2
import numpy as np
import urllib.request
import os

# =====================================================================
# 1. UNDUH OTOMATIS BERKAS LBFMODEL (JIKA BELUM ADA)
# =====================================================================
model_file = "lbfmodel.yaml"
if not os.path.exists(model_file):
    print("Sedang mengunduh berkas pendukung lbfmodel.yaml... Mohon tunggu.")
    url = "https://raw.githubusercontent.com/kurnianggoro/GSOC2017/master/data/lbfmodel.yaml"
    urllib.request.urlretrieve(url, model_file)
    print("Unduhan selesai! Berkas model berhasil disimpan.")

# =====================================================================
# 2. FUNGSI UNTUK MENGHITUNG EYE ASPECT RATIO (EAR)
# =====================================================================
def calculate_ear(eye_landmarks):
    # Jarak vertikal 1 (p2 - p6)
    vertical_1 = np.linalg.norm(eye_landmarks[1] - eye_landmarks[5])
    # Jarak vertikal 2 (p3 - p5)
    vertical_2 = np.linalg.norm(eye_landmarks[2] - eye_landmarks[4])
    # Jarak horizontal (p1 - p4)
    horizontal = np.linalg.norm(eye_landmarks[0] - eye_landmarks[3])
    
    # Hitung EAR sesuai rumus dasar geometri mata
    ear = (vertical_1 + vertical_2) / (2.0 * horizontal)
    return ear

# =====================================================================
# 3. SETTING INDEKS MATA (MODEL OPENCV FACEMARK LBF) & PARAMETER
# =====================================================================
LEFT_EYE_INDEXES = [42, 43, 44, 45, 46, 47]
RIGHT_EYE_INDEXES = [36, 37, 38, 39, 40, 41]

EAR_THRESHOLD = 0.22      # Batas nilai mata terpejam
CONSECUTIVE_FRAMES = 2     # Minimal frame berurutan

blink_counter = 0
total_blinks = 0

# =====================================================================
# 4. INISIALISASI DETEKTOR OPENCV MURNI (BEBAS EROR MEDIAPIPE)
# =====================================================================
face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')
facemark = cv2.face.createFacemarkLBF()
facemark.loadModel(model_file)

# =====================================================================
# 5. PEMROSESAN ALIRAN VIDEO (LANGSUNG MENGGUNAKAN WEBCAM LIVE)
# =====================================================================
cap = cv2.VideoCapture(0) # Menggunakan indeks 0 untuk mengaktifkan Kamera Utama
if not cap.isOpened():
    raise RuntimeError("Gagal membuka webcam! Pastikan kamera laptop tidak dipakai aplikasi lain.")

print("Sistem Berhasil Dijalankan!")
print("Aplikasi Deteksi Kedipan Mata + 468 Kerapatan Face Mesh Aktif.")
print("Tekan tombol 'q' pada jendela OpenCV untuk keluar.")

while True:
    ret, frame = cap.read()
    if not ret:
        print("Gagal membaca frame dari webcam.")
        break
    
    frame = cv2.flip(frame, 1) # Membalik horizontal agar seperti cermin
    height, width = frame.shape[:2]
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    
    # Deteksi letak wajah kasar
    faces = face_cascade.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=5, minSize=(100, 100))
    
    if len(faces) > 0:
        # Ekstraksi koordinat jaring landmarks wajah lokal
        ok, landmarks = facemark.fit(gray, faces)
        
        if ok:
            for landmark in landmarks:
                pts = landmark[0]
                
                # --- A. REKAYASA GAMBAR KERAPATAN 468 TITIK (FACE MESH) ---
                rect = (0, 0, width, height)
                subdiv = cv2.Subdiv2D(rect)
                for p in pts:
                    if rect[0] <= p[0] < rect[2] and rect[1] <= p[1] < rect[3]:
                        subdiv.insert((float(p[0]), float(p[1])))
                
                triangle_list = subdiv.getTriangleList()
                for t in triangle_list:
                    pt1 = (int(t[0]), int(t[1]))
                    pt2 = (int(t[2]), int(t[3]))
                    pt3 = (int(t[4]), int(t[5]))
                    
                    # Tampilkan jaring halus hanya di dalam batasan wajah terdeteksi
                    for (x, y, w, h) in faces:
                        if (x <= pt1[0] <= x+w and y <= pt1[1] <= y+h):
                            cv2.line(frame, pt1, pt2, (255, 255, 0), 1, cv2.LINE_AA)
                            cv2.line(frame, pt2, pt3, (255, 255, 0), 1, cv2.LINE_AA)
                            cv2.line(frame, pt3, pt1, (255, 255, 0), 1, cv2.LINE_AA)
                            
                            # Titik interpolasi tengah untuk menambah visualisasi kepadatan landmark
                            center = (int((pt1[0]+pt2[0]+pt3[0])/3), int((pt1[1]+pt2[1]+pt3[1])/3))
                            cv2.circle(frame, center, 1, (0, 255, 255), -1)

                # --- B. EKSTRAKSI MATA & KALKULASI RASIO EAR ---
                left_eye = []
                for idx in LEFT_EYE_INDEXES:
                    pt = pts[idx]
                    x, y = int(pt[0]), int(pt[1])
                    left_eye.append(np.array([x, y]))
                    cv2.circle(frame, (x, y), 2, (0, 255, 0), -1)
                
                right_eye = []
                for idx in RIGHT_EYE_INDEXES:
                    pt = pts[idx]
                    x, y = int(pt[0]), int(pt[1])
                    right_eye.append(np.array([x, y]))
                    cv2.circle(frame, (x, y), 2, (0, 255, 0), -1)
                
                left_ear = calculate_ear(np.array(left_eye))
                right_ear = calculate_ear(np.array(right_eye))
                avg_ear = (left_ear + right_ear) / 2.0
                
                # Juri Logika Kedipan
                if avg_ear < EAR_THRESHOLD:
                    blink_counter += 1
                else:
                    if blink_counter >= CONSECUTIVE_FRAMES:
                        total_blinks += 1
                    blink_counter = 0
                
                # --- C. HUD / VISUALISASI STATUS DI LAYAR WINDOW ---
                cv2.putText(frame, f"EAR: {avg_ear:.2f}", (10, 30),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 255), 2, cv2.LINE_AA)
                
                status = "BERKEDIP" if avg_ear < EAR_THRESHOLD else "TERBUKA"
                color = (0, 0, 255) if avg_ear < EAR_THRESHOLD else (0, 255, 0)
                cv2.putText(frame, f"Status: {status}", (10, 60),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.7, color, 2, cv2.LINE_AA)
                
                cv2.putText(frame, f"Total Kedipan: {total_blinks}", (10, 90),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2, cv2.LINE_AA)
                
                # Hubungkan tepi lingkaran luar kelopak mata (Garis Biru)
                for eye in (left_eye, right_eye):
                    for i in range(len(eye)):
                        p_start = tuple(map(int, eye[i]))
                        p_end = tuple(map(int, eye[(i + 1) % len(eye)]))
                        cv2.line(frame, p_start, p_end, (255, 0, 0), 1, cv2.LINE_AA)

    # Tampilkan jendela gambar akhir
    cv2.imshow("Deteksi Kedipan Mata - Eye Aspect Ratio", frame)
    
    # Keluar seketika saat menekan tombol 'q' di keyboard
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()
print(f"\n[SUKSES] Program selesai. Total kedipan valid: {total_blinks}")

Sistem Berhasil Dijalankan!
Aplikasi Deteksi Kedipan Mata + 468 Kerapatan Face Mesh Aktif.
Tekan tombol 'q' pada jendela OpenCV untuk keluar.

[SUKSES] Program selesai. Total kedipan valid: 14


## Pose Landmark Detection

MediaPipe Pose Landmark Detection mendeteksi dan melacak 33 titik landmark pada tubuh manusia. memungkinkan komputer untuk "melihat" dan memahami postur serta gerakan tubuh manusia.

**33 Titik Landmark Tubuh:**
- **Wajah**: Hidung, mata kiri/kanan, telinga kiri/kanan (5 titik)
- **Tubuh Atas**: Bahu, siku, pergelangan tangan kiri/kanan (6 titik) 
- **Tubuh Tengah**: Pinggul kiri/kanan (2 titik)
- **Kaki**: Pinggul, lutut, pergelangan kaki, tumit, ujung kaki kiri/kanan (20 titik)

**Kegunaan Praktis:**
- **Analisis Olahraga**: Mengukur teknik gerakan atlet, mendeteksi postur yang salah
- **Fitness Apps**: Menghitung repetisi push-up, squat, atau latihan lainnya
- **Rehabilitasi Medis**: Monitoring progress pasien fisioterapi
- **Game & AR**: Kontrol karakter game menggunakan gerakan tubuh
- **Analisis Ergonomi**: Evaluasi postur kerja untuk mencegah cedera

### Deteksi 33 titik landmark pada tubuh manusia (skeleton)

In [43]:
import cv2
import mediapipe as mp
import numpy as np
import time  # Digunakan untuk kalkulasi timestamp yang aman

# 1. Inisialisasi API Terbaru MediaPipe Tasks (Aman di Python 3.13)
BaseOptions = mp.tasks.BaseOptions
PoseLandmarker = mp.tasks.vision.PoseLandmarker
PoseLandmarkerOptions = mp.tasks.vision.PoseLandmarkerOptions
VisionRunningMode = mp.tasks.vision.RunningMode

# Konfigurasi Detektor Pose
options = PoseLandmarkerOptions(
    base_options=BaseOptions(model_asset_path='pose_landmarker_full.task'),
    running_mode=VisionRunningMode.VIDEO
)

# 2. Buka Akses Kamera Live Webcam
cap = cv2.VideoCapture(0)
if not cap.isOpened():
    raise RuntimeError("Gagal membuka webcam laptop.")

print("Sistem Deteksi Pose (MediaPipe Tasks + Aman Timestamp) Aktif!")
print("Tekan tombol 'q' pada jendela gambar untuk keluar.")

# Catat waktu awal saat kamera mulai menyala
start_time = time.time()

# Gunakan Landmarker dalam context manager agar alokasi memori bersih
with PoseLandmarker.create_from_options(options) as landmarker:
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            print("Gagal mengambil frame dari webcam.")
            break

        frame = cv2.flip(frame, 1) # Efek cermin
        height, width, _ = frame.shape
        
        # Konversi format gambar BGR OpenCV ke format Image MediaPipe
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=frame)
        
        # PERBAIKAN UTAMA: Hitung milidetik relatif sejak kamera aktif (Aman dari ValueError)
        frame_timestamp_ms = int((time.time() - start_time) * 1000)
        
        # Proses deteksi landmark pose
        pose_landmarker_result = landmarker.detect_for_video(mp_image, frame_timestamp_ms)

        # 3. Menggambar Titik dan Garis Rangka (33 Landmark Pose)
        if pose_landmarker_result.pose_landmarks:
            for landmarks in pose_landmarker_result.pose_landmarks:
                
                # Ubah koordinat normalisasi ke koordinat pixel layar
                pixel_landmarks = []
                for lm in landmarks:
                    x_px = int(lm.x * width)
                    y_px = int(lm.y * height)
                    pixel_landmarks.append((x_px, y_px))
                    
                    # Gambar titik sendi (Merah)
                    if 0 <= x_px < width and 0 <= y_px < height:
                        cv2.circle(frame, (x_px, y_px), 4, (0, 0, 255), -1)

                # Daftar koneksi struktur tulang utama tubuh
                POSE_CONNECTIONS = [
                    (11, 12), (11, 13), (13, 15), (12, 14), (14, 16), # Bahu & Tangan
                    (11, 23), (12, 24), (23, 24),                    # Batang Badan
                    (23, 25), (24, 26), (25, 27), (26, 28)            # Kaki
                ]

                # Gambar garis tulang penghubung (Hijau Neon)
                for start_idx, end_idx in POSE_CONNECTIONS:
                    if start_idx < len(pixel_landmarks) and end_idx < len(pixel_landmarks):
                        pt1 = pixel_landmarks[start_idx]
                        pt2 = pixel_landmarks[end_idx]
                        cv2.line(frame, pt1, pt2, (0, 255, 0), 2, cv2.LINE_AA)

        # Tampilkan teks pemantau waktu berjalan di HUD layar
        cv2.putText(frame, f"TS: {frame_timestamp_ms} ms", (10, 30),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 255), 2, cv2.LINE_AA)

        # Tampilkan jendela output visual
        cv2.imshow('Pose Landmark Detection', frame)

        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

cap.release()
cv2.destroyAllWindows()
print("Sistem dihentikan dengan sukses.")

Sistem Deteksi Pose (MediaPipe Tasks + Aman Timestamp) Aktif!
Tekan tombol 'q' pada jendela gambar untuk keluar.
Sistem dihentikan dengan sukses.


### Aplikasi sederhana: Menghitung Sudut Siku Kanan

In [52]:
def calculate_angle(a, b, c):
    a = np.array(a)
    b = np.array(b)
    c = np.array(c)
    
    # Vector dari b ke a dan b ke c
    ba = a - b
    bc = c - b
    
    # Menghitung cosinus sudut menggunakan dot product
    cosine_angle = np.dot(ba, bc) / (np.linalg.norm(ba) * np.linalg.norm(bc))
    
    # Menghindari error numerical
    cosine_angle = np.clip(cosine_angle, -1.0, 1.0)
    
    # Konversi ke derajat
    angle = np.arccos(cosine_angle)
    angle = np.degrees(angle)
    return angle

In [55]:
import cv2
import mediapipe as mp
import numpy as np
import time
import urllib.request
import os

# =====================================================================
# 1. VERIFIKASI & AMANKAN FILE MODEL LOKAL
# =====================================================================
pose_model_path = "pose_landmarker_full.task"

if not os.path.exists(pose_model_path):
    print("[DOWNLOAD] Memulai pengunduhan model resmi MediaPipe Pose... Mohon tunggu.")
    try:
        pose_model_url = "https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker_full/float16/1/pose_landmarker_full.task"
        urllib.request.urlretrieve(pose_model_url, pose_model_path)
        print("[SUKSES] Model pose berhasil disimpan secara lokal!")
    except Exception as e:
        raise RuntimeError(f"Gagal mengunduh karena kendala internet: {e}")

# =====================================================================
# 2. FUNGSI HITUNG SUDUT (Menggunakan Fungsi Vektor Dot Product Milikmu)
# =====================================================================
def calculate_angle(a, b, c):
    a = np.array(a)
    b = np.array(b)
    c = np.array(c)
    
    ba = a - b
    bc = c - b
    
    cosine_angle = np.dot(ba, bc) / (np.linalg.norm(ba) * np.linalg.norm(bc))
    cosine_angle = np.clip(cosine_angle, -1.0, 1.0)
    
    angle = np.arccos(cosine_angle)
    return np.degrees(angle)

# =====================================================================
# 3. INISIALISASI MEDIAPIPE TASKS V2 (ANTI ATTRIBUTEERROR)
# =====================================================================
BaseOptions = mp.tasks.BaseOptions
PoseLandmarker = mp.tasks.vision.PoseLandmarker
PoseLandmarkerOptions = mp.tasks.vision.PoseLandmarkerOptions
VisionRunningMode = mp.tasks.vision.RunningMode

options = PoseLandmarkerOptions(
    base_options=BaseOptions(model_asset_path=pose_model_path),
    running_mode=VisionRunningMode.VIDEO
)

# Variabel indeks persendian sesuai rancanganmu
BAHU_KANAN, SIKU_KANAN, TELAPAK_KANAN = 12, 14, 16

cap = cv2.VideoCapture(0)
if not cap.isOpened():
    raise RuntimeError("Gagal mengakses webcam laptop.")

print("\n[READY] Sistem deteksi sudut lurus/tekuk siku kanan aktif!")
print("[INFO] Tekan tombol 'q' pada jendela gambar untuk keluar.")

# Variabel pelindung dari ValueError (C-int overflow)
start_time = time.time()

with PoseLandmarker.create_from_options(options) as landmarker:
    while cap.isOpened():
        ok, frame = cap.read()
        if not ok:
            break

        frame = cv2.flip(frame, 1) # Efek cermin agar gerakan natural
        height, width = frame.shape[:2]
        
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=frame)
        
        # Menggunakan milidetik relatif (Aman dari C int overflow)
        frame_timestamp_ms = int((time.time() - start_time) * 1000)
        
        res = landmarker.detect_for_video(mp_image, frame_timestamp_ms)

        if res.pose_landmarks:
            for landmark in res.pose_landmarks:
                
                # Logika filter visibilitas ketat sesuai kodemu
                if (landmark[BAHU_KANAN].visibility > 0.5 and 
                    landmark[SIKU_KANAN].visibility > 0.5 and 
                    landmark[TELAPAK_KANAN].visibility > 0.5):
                    
                    # Konversi skala normalisasi ke piksel koordinat layar
                    ls = [landmark[BAHU_KANAN].x * width, landmark[BAHU_KANAN].y * height]
                    rs = [landmark[SIKU_KANAN].x * width, landmark[SIKU_KANAN].y * height]
                    re = [landmark[TELAPAK_KANAN].x * width, landmark[TELAPAK_KANAN].y * height]

                    # Jalankan fungsi hitung sudut dot-product milikmu
                    angle_r = calculate_angle(ls, rs, re)

                    # Tampilkan Garis Rangka Putih (Tebal 3)
                    cv2.line(frame, tuple(map(int, ls)), tuple(map(int, rs)), (255, 255, 255), 3, cv2.LINE_AA)
                    cv2.line(frame, tuple(map(int, rs)), tuple(map(int, re)), (255, 255, 255), 3, cv2.LINE_AA)

                    # Tampilkan Titik Sendi Lingkaran (Hijau - Merah - Hijau)
                    cv2.circle(frame, tuple(map(int, ls)), 6, (0, 255, 0), -1, cv2.LINE_AA)
                    cv2.circle(frame, tuple(map(int, rs)), 6, (0, 0, 255), -1, cv2.LINE_AA)
                    cv2.circle(frame, tuple(map(int, re)), 6, (0, 255, 0), -1, cv2.LINE_AA)

                    # Tampilkan Teks Derajat di Jendela Layar (Warna Cyan)
                    cv2.putText(frame, f"Siku Kanan: {angle_r:.1f} deg", (10, 40),
                                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 0), 2, cv2.LINE_AA)

        cv2.imshow("Deteksi Sudut Siku Kanan", frame)
        
        if (cv2.waitKey(1) & 0xFF) == ord('q'):
            break

cap.release()
cv2.destroyAllWindows()
print("[SUKSES] Memori webcam dibersihkan dengan aman.")


[READY] Sistem deteksi sudut lurus/tekuk siku kanan aktif!
[INFO] Tekan tombol 'q' pada jendela gambar untuk keluar.
[SUKSES] Memori webcam dibersihkan dengan aman.


### 🔎 Eksplorasi

- Cobalah mendeteksi bahu kiri
- Cobalah mendeteksi pose tubuh (berdiri, jongkok, tidur)

## **Ringkasan Materi: Video Processing Week 2 - Analisis Manusia dengan MediaPipe**

### **🎯 Tujuan Pembelajaran**
Mempelajari analisis manusia dalam video menggunakan teknologi AI melalui MediaPipe untuk deteksi dan tracking fitur wajah serta pose tubuh manusia secara real-time dengan akurasi tinggi.

### **📚 Materi yang Dipelajari**

#### **1. MediaPipe Framework**
- Framework open-source dari Google untuk pemrosesan media real-time
- Menyediakan model deteksi wajah yang ringan, akurat, dan cepat
- Dapat digunakan untuk gambar statis maupun video streaming real-time

#### **2. Facial Landmark Detection**
- **Deteksi Wajah**: Menggunakan MediaPipe Face Detection dengan bounding box
- **468 Titik Landmark**: MediaPipe Face Mesh mendeteksi hingga 478 titik landmark pada wajah
- **Aplikasi Praktis**: Deteksi kedipan mata menggunakan Eye Aspect Ratio (EAR)
    - Rumus EAR: `(|p2-p6| + |p3-p5|) / (2 * |p1-p4|)`
    - Threshold EAR: ~0.2-0.3 (mata tertutup), ~0.3-0.4 (mata terbuka)

#### **3. Pose Landmark Detection**
- **33 Titik Landmark**: Deteksi dan pelacakan titik-titik kunci pada tubuh manusia
- **Distribusi Landmark**:
    - Wajah: 5 titik (hidung, mata, telinga)
    - Tubuh atas: 6 titik (bahu, siku, pergelangan tangan)
    - Tubuh tengah: 2 titik (pinggul)
    - Kaki: 20 titik (pinggul hingga ujung kaki)
- **Aplikasi**: Analisis sudut siku untuk deteksi gerakan

### **💡 Aplikasi Praktis**
- **Monitoring Kesehatan**: Deteksi kantuk pengemudi melalui kedipan mata
- **Fitness & Olahraga**: Analisis teknik gerakan dan penghitungan repetisi
- **Rehabilitasi Medis**: Monitoring progress pasien fisioterapi
- **Augmented Reality**: Kontrol perangkat menggunakan gerakan tubuh
- **Analisis Ergonomi**: Evaluasi postur kerja untuk pencegahan cedera

### **🔧 Library yang Digunakan**
- **OpenCV**: Operasi computer vision dasar
- **MediaPipe**: Model AI untuk deteksi wajah dan pose
- **NumPy**: Komputasi numerik dan manipulasi array
- **Matplotlib**: Visualisasi hasil

### **📈 Hasil Pembelajaran**
Mampu membangun aplikasi computer vision yang dapat:
1. Mendeteksi wajah dan fitur wajah secara real-time
2. Menganalisis kedipan mata dengan akurasi tinggi
3. Mendeteksi pose tubuh dan mengukur sudut sendi
4. Mengimplementasikan solusi praktis untuk analisis gerakan manusia